[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-14-capstone-pipeline.ipynb#scrollTo=a1e2f3b4)

---
# Day 14 · Capstone — Full MLOps Pipeline: Train → Track → Register → Serve
**certified-journeys / mlflow-certified** · Capstone · End-to-End MLOps

> **Goal for today:** Build a complete, reproducible MLOps pipeline: train three model variants with full MLflow tracking, promote the best to Production in the Model Registry, package a preprocessing + model Pipeline as a pyfunc artifact, serve it locally via `mlflow models serve`, and log a summary report run.

---

## Capstone architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                     MLflow Tracking Server (SQLite)             │
│                                                                 │
│  Experiment: capstone-pipeline                                  │
│  ├── run: LogisticRegression      (params + metrics + model)    │
│  ├── run: RandomForest            (params + metrics + model)    │
│  ├── run: GradientBoosting        (params + metrics + model)    │
│  └── run: pipeline-report (summary artifact + champion metrics) │
│                                                                 │
│  Model Registry: breast-cancer-pipeline                         │
│  ├── v1 (LogisticRegression)     stage: Archived                │
│  ├── v2 (RandomForest)           stage: Archived  (or Prod)     │
│  └── v3 (GradientBoosting)       alias: champion → Production   │
└─────────────────────────────────────────────────────────────────┘
                              ↓
           mlflow models serve  (REST API on localhost:5001)
                              ↓
           POST /invocations   { "dataframe_split": {...} }
```

Each section below builds one layer of this pipeline.


In [ ]:
%pip install -q mlflow scikit-learn pandas numpy requests


## Step 1 · Environment Setup and Dataset

We use the **Breast Cancer Wisconsin** dataset — a real binary classification problem (malignant / benign) with 30 numeric features. This dataset is built into scikit-learn and requires no downloads.

| Dataset property | Value |
|---|---|
| Samples | 569 |
| Features | 30 (all numeric, no missing values) |
| Classes | 2 (0 = malignant, 1 = benign) |
| Baseline (majority class) | ~62.7% |

A **SQLite tracking URI** gives us a full-featured MLflow backend (including Model Registry) without running a separate server process — ideal for this capstone.


In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature

import pandas as pd
import numpy as np
import json
import os
import tempfile
import subprocess
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
    classification_report
)

# ── Tracking setup ─────────────────────────────────────────────────────────
DB_PATH = "sqlite:///capstone_pipeline.db"
mlflow.set_tracking_uri(DB_PATH)
EXPERIMENT_NAME = "capstone-pipeline"
mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient()
MODEL_REGISTRY_NAME = "breast-cancer-pipeline"

# ── Dataset ────────────────────────────────────────────────────────────────
bc = load_breast_cancer(as_frame=True)
X, y = bc.data, bc.target
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("=== Dataset ===")
print(f"  Total samples  : {len(X)}")
print(f"  Train / Test   : {len(X_train)} / {len(X_test)}")
print(f"  Features       : {X.shape[1]}")
print(f"  Class balance  : {y.value_counts().to_dict()}")
print(f"  Majority baseline: {y.value_counts(normalize=True).max():.3f}")
print(f"\nMLflow tracking URI : {DB_PATH}")
print(f"Experiment name     : {EXPERIMENT_NAME}")


**What just happened?**
- We configured a **single SQLite backend** — this is a common local MLOps setup; in production you'd point `mlflow.set_tracking_uri` at a dedicated MLflow server with PostgreSQL.
- **`stratify=y`** in `train_test_split` preserves class proportions in both splits — critical for imbalanced datasets.
- We defined `MODEL_REGISTRY_NAME` once so all registry operations use the same string consistently.


## Step 2 · Train Three Model Variants with Full MLflow Tracking

Each variant gets its own MLflow run with:
- **`log_params`** — hyperparameters (reproducibility)
- **`log_metrics`** — accuracy, F1, ROC-AUC, precision, recall
- **`log_model`** with signature + input example (serving readiness)
- **`set_tags`** — provenance metadata

We use `cross_val_score` for a more reliable CV estimate alongside the held-out test metrics.

| Model | Strengths | Weaknesses |
|---|---|---|
| `LogisticRegression` | Fast, interpretable, well-calibrated | Linear decision boundary |
| `RandomForestClassifier` | Handles non-linearity, feature importance | Slower, more memory |
| `GradientBoostingClassifier` | Often best accuracy, robust to outliers | Slowest to train, many hyperparams |


In [ ]:
import sklearn

def get_repro_tags():
    """Return a dict of reproducibility tags for any run."""
    try:
        commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        commit = "unknown"
    return {
        "git.commit":          commit,
        "data.source":         "sklearn.datasets.load_breast_cancer",
        "data.n_samples":      str(len(X)),
        "data.test_size":      "0.20",
        "env.mlflow_version":  mlflow.__version__,
        "env.sklearn_version": sklearn.__version__,
        "pipeline.stage":      "training",
    }

def evaluate_and_log(clf, X_tr, X_te, y_tr, y_te, params: dict, run_name: str):
    """
    Fit clf, log full metrics + model to MLflow, return run_id and test metrics.
    Must be called INSIDE an active mlflow.start_run() context.
    """
    clf.fit(X_tr, y_tr)
    preds      = clf.predict(X_te)
    proba      = clf.predict_proba(X_te)[:, 1] if hasattr(clf, "predict_proba") else preds

    metrics = {
        "accuracy":  accuracy_score(y_te, preds),
        "f1":        f1_score(y_te, preds),
        "roc_auc":   roc_auc_score(y_te, proba),
        "precision": precision_score(y_te, preds),
        "recall":    recall_score(y_te, preds),
    }

    # 5-fold CV on training data for a less optimistic estimate
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=5, scoring="roc_auc")
    metrics["cv_roc_auc_mean"] = float(cv_scores.mean())
    metrics["cv_roc_auc_std"]  = float(cv_scores.std())

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.set_tags(get_repro_tags())
    mlflow.set_tag("model.class", type(clf).__name__)

    signature = infer_signature(X_te, preds)
    mlflow.sklearn.log_model(
        sk_model=clf,
        artifact_path="model",
        signature=signature,
        input_example=X_te.head(3),
    )

    return metrics


# ── Run 1: LogisticRegression ──────────────────────────────────────────────
lr_params = {"C": 1.0, "max_iter": 1000, "solver": "lbfgs", "random_state": 42}
with mlflow.start_run(run_name="LogisticRegression") as run_lr:
    lr_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(**lr_params)),
    ])
    # Note: log the outer Pipeline as the model so scaler is bundled
    lr_clf.fit(X_train, y_train)
    preds_lr   = lr_clf.predict(X_test)
    proba_lr   = lr_clf.predict_proba(X_test)[:, 1]
    lr_metrics = {
        "accuracy":  accuracy_score(y_test, preds_lr),
        "f1":        f1_score(y_test, preds_lr),
        "roc_auc":   roc_auc_score(y_test, proba_lr),
        "precision": precision_score(y_test, preds_lr),
        "recall":    recall_score(y_test, preds_lr),
    }
    cv_lr = cross_val_score(lr_clf, X_train, y_train, cv=5, scoring="roc_auc")
    lr_metrics["cv_roc_auc_mean"] = float(cv_lr.mean())
    lr_metrics["cv_roc_auc_std"]  = float(cv_lr.std())

    mlflow.log_params(lr_params)
    mlflow.log_metrics(lr_metrics)
    mlflow.set_tags(get_repro_tags())
    mlflow.set_tag("model.class", "LogisticRegression")
    sig_lr = infer_signature(X_test, preds_lr)
    mlflow.sklearn.log_model(lr_clf, "model", signature=sig_lr, input_example=X_test.head(3))
    lr_run_id = run_lr.info.run_id

print(f"LogisticRegression  | acc={lr_metrics['accuracy']:.4f} | roc_auc={lr_metrics['roc_auc']:.4f} | run={lr_run_id[:8]}…")


# ── Run 2: RandomForest ───────────────────────────────────────────────────
rf_params = {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 2, "random_state": 42}
with mlflow.start_run(run_name="RandomForest") as run_rf:
    rf_clf = RandomForestClassifier(**rf_params)
    rf_clf.fit(X_train, y_train)
    preds_rf   = rf_clf.predict(X_test)
    proba_rf   = rf_clf.predict_proba(X_test)[:, 1]
    rf_metrics = {
        "accuracy":  accuracy_score(y_test, preds_rf),
        "f1":        f1_score(y_test, preds_rf),
        "roc_auc":   roc_auc_score(y_test, proba_rf),
        "precision": precision_score(y_test, preds_rf),
        "recall":    recall_score(y_test, preds_rf),
    }
    cv_rf = cross_val_score(rf_clf, X_train, y_train, cv=5, scoring="roc_auc")
    rf_metrics["cv_roc_auc_mean"] = float(cv_rf.mean())
    rf_metrics["cv_roc_auc_std"]  = float(cv_rf.std())

    mlflow.log_params(rf_params)
    mlflow.log_metrics(rf_metrics)
    mlflow.set_tags(get_repro_tags())
    mlflow.set_tag("model.class", "RandomForest")
    sig_rf = infer_signature(X_test, preds_rf)
    mlflow.sklearn.log_model(rf_clf, "model", signature=sig_rf, input_example=X_test.head(3))
    rf_run_id = run_rf.info.run_id

print(f"RandomForest        | acc={rf_metrics['accuracy']:.4f} | roc_auc={rf_metrics['roc_auc']:.4f} | run={rf_run_id[:8]}…")


# ── Run 3: GradientBoosting ───────────────────────────────────────────────
gb_params = {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 4, "random_state": 42}
with mlflow.start_run(run_name="GradientBoosting") as run_gb:
    gb_clf = GradientBoostingClassifier(**gb_params)
    gb_clf.fit(X_train, y_train)
    preds_gb   = gb_clf.predict(X_test)
    proba_gb   = gb_clf.predict_proba(X_test)[:, 1]
    gb_metrics = {
        "accuracy":  accuracy_score(y_test, preds_gb),
        "f1":        f1_score(y_test, preds_gb),
        "roc_auc":   roc_auc_score(y_test, proba_gb),
        "precision": precision_score(y_test, preds_gb),
        "recall":    recall_score(y_test, preds_gb),
    }
    cv_gb = cross_val_score(gb_clf, X_train, y_train, cv=5, scoring="roc_auc")
    gb_metrics["cv_roc_auc_mean"] = float(cv_gb.mean())
    gb_metrics["cv_roc_auc_std"]  = float(cv_gb.std())

    mlflow.log_params(gb_params)
    mlflow.log_metrics(gb_metrics)
    mlflow.set_tags(get_repro_tags())
    mlflow.set_tag("model.class", "GradientBoosting")
    sig_gb = infer_signature(X_test, preds_gb)
    mlflow.sklearn.log_model(gb_clf, "model", signature=sig_gb, input_example=X_test.head(3))
    gb_run_id = run_gb.info.run_id

print(f"GradientBoosting    | acc={gb_metrics['accuracy']:.4f} | roc_auc={gb_metrics['roc_auc']:.4f} | run={gb_run_id[:8]}…")


**What just happened?**
- **Three separate MLflow runs** — each is independently queryable, comparable, and auditable. In a real CI pipeline, each would be a separate job with its own environment.
- **`cross_val_score` on training data** gives a less optimistic estimate than test-set metrics alone — `cv_roc_auc_mean` is what we use to pick the champion (not just test accuracy).
- **`infer_signature(X_test, preds)`** captures the exact DataFrame schema: column names, dtypes, and output format. MLflow Serving enforces this schema at the REST endpoint.
- **LogisticRegression is wrapped in a Pipeline** with a StandardScaler — this ensures the scaler is bundled with the model and applied consistently at serving time.


## Step 3 · Select the Best Model and Promote to Production

Champion selection criteria (in priority order):
1. **ROC-AUC** on the held-out test set — primary metric for medical classification
2. **CV ROC-AUC mean** — secondary tiebreaker (more robust than single-split)
3. **F1 score** — accounts for class imbalance

After selecting the champion, we:
1. Register all three models in the Model Registry (for audit completeness)
2. Promote the champion to `Production` and archive the others
3. Set `@champion` alias on the production version


In [ ]:
# Build a leaderboard to pick the champion
candidates = [
    {"name": "LogisticRegression", "run_id": lr_run_id, "metrics": lr_metrics},
    {"name": "RandomForest",       "run_id": rf_run_id, "metrics": rf_metrics},
    {"name": "GradientBoosting",   "run_id": gb_run_id, "metrics": gb_metrics},
]

leaderboard = pd.DataFrame([
    {
        "model":           c["name"],
        "roc_auc":         c["metrics"]["roc_auc"],
        "cv_roc_auc_mean": c["metrics"]["cv_roc_auc_mean"],
        "accuracy":        c["metrics"]["accuracy"],
        "f1":              c["metrics"]["f1"],
        "run_id":          c["run_id"],
    }
    for c in candidates
]).sort_values("roc_auc", ascending=False).reset_index(drop=True)

print("=== Champion Selection Leaderboard ===")
print(leaderboard[["model", "roc_auc", "cv_roc_auc_mean", "accuracy", "f1"]].to_string(index=True))

# Select champion by highest test ROC-AUC
champion_row = leaderboard.iloc[0]
champion_name = champion_row["model"]
champion_run_id = champion_row["run_id"]
champion_metrics = {k: float(champion_row[k]) for k in ["roc_auc", "cv_roc_auc_mean", "accuracy", "f1"]}

print(f"\nChampion: {champion_name} (run={champion_run_id[:8]}…)")
print(f"  roc_auc         : {champion_metrics['roc_auc']:.4f}")
print(f"  cv_roc_auc_mean : {champion_metrics['cv_roc_auc_mean']:.4f}")


In [ ]:
# Register all three candidates, then promote champion
registered_versions = {}

for c in candidates:
    model_uri = f"runs:/{c['run_id']}/model"
    mv = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)
    registered_versions[c["name"]] = mv
    # Add a description with key metrics so the registry is self-documenting
    client.update_model_version(
        name=MODEL_REGISTRY_NAME,
        version=mv.version,
        description=(
            f"{c['name']} | "
            f"roc_auc={c['metrics']['roc_auc']:.4f} | "
            f"accuracy={c['metrics']['accuracy']:.4f} | "
            f"run_id={c['run_id'][:8]}"
        ),
    )
    print(f"Registered {c['name']:22s} → v{mv.version}")

# Promote champion to Production, archive the rest
champion_mv = registered_versions[champion_name]
client.transition_model_version_stage(
    name=MODEL_REGISTRY_NAME,
    version=champion_mv.version,
    stage="Production",
    archive_existing_versions=True,  # archive any prior Production versions
)
client.set_registered_model_alias(
    name=MODEL_REGISTRY_NAME,
    alias="champion",
    version=champion_mv.version,
)

# Archive non-champion versions
for name, mv in registered_versions.items():
    if name != champion_name:
        client.transition_model_version_stage(
            name=MODEL_REGISTRY_NAME,
            version=mv.version,
            stage="Archived",
        )

print(f"\n✓ Champion '{champion_name}' promoted to Production (v{champion_mv.version})")
print(f"  @champion alias → v{champion_mv.version}")

# Print final registry state
print("\n=== Registry state ===")
all_versions = client.search_model_versions(f"name='{MODEL_REGISTRY_NAME}'")
for v in sorted(all_versions, key=lambda x: int(x.version)):
    print(f"  v{v.version} | {v.current_stage:12s} | {v.description[:65] if v.description else ''}")


**What just happened?**
- We registered **all three models** — even the non-champions — because the registry is the authoritative audit trail. You can always look up why a version wasn't promoted.
- **`archive_existing_versions=True`** ensures exactly one Production version at any time — a safety invariant your serving infrastructure can rely on.
- **`@champion` alias** decouples the serving URI from the version number. Your inference server loads `models:/breast-cancer-pipeline@champion` and never needs to know the version integer.
- **`update_model_version(description=...)`** is essential in a team setting — the next engineer reading the registry knows immediately what each version is and why it was (or wasn't) promoted.


## Step 4 · Build and Log a Full Preprocessing + Model Pipeline as pyfunc

The **`mlflow.pyfunc`** flavour lets you package arbitrary Python logic — not just a single estimator — as a deployable model artifact. This is the production-grade pattern when your inference logic includes:
- Pre-processing (scaling, feature selection)
- Post-processing (threshold tuning, output formatting)
- Business logic (e.g. returning human-readable labels)

| Approach | Use case |
|---|---|
| `mlflow.sklearn.log_model` | Simple: the artifact IS the sklearn object |
| `mlflow.pyfunc.log_model` | Advanced: wrap any Python object with custom `predict()` |

The `PythonModel` subclass approach requires implementing a `predict(context, model_input)` method. MLflow calls this at serving time.


In [ ]:
# Build the full preprocessing + classifier pipeline using sklearn Pipeline
full_pipeline = Pipeline([
    ("scaler",   StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=15)),  # select top 15 features
    ("clf",      GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                            max_depth=4, random_state=42)),
])
full_pipeline.fit(X_train, y_train)

# Verify the pipeline performs well before logging it
pipeline_preds = full_pipeline.predict(X_test)
pipeline_proba = full_pipeline.predict_proba(X_test)[:, 1]
pipeline_metrics = {
    "accuracy": accuracy_score(y_test, pipeline_preds),
    "f1":       f1_score(y_test, pipeline_preds),
    "roc_auc":  roc_auc_score(y_test, pipeline_proba),
}
print("Full pipeline (scaler + selector + GBM):")
for k, v in pipeline_metrics.items():
    print(f"  {k}: {v:.4f}")

# ── Custom pyfunc wrapper ──────────────────────────────────────────────────
# We wrap the pipeline in a PythonModel so we can add business logic:
# - Return both the binary prediction AND the malignant probability
# - Return human-readable class labels instead of 0/1

import mlflow.pyfunc

class CancerPipelineModel(mlflow.pyfunc.PythonModel):
    """
    Production pyfunc wrapper around a sklearn Pipeline.
    Adds human-readable labels and probability outputs.
    """

    def load_context(self, context):
        """Load the serialised pipeline from the artifact store."""
        import pickle
        with open(context.artifacts["pipeline"], "rb") as f:
            self.pipeline = pickle.load(f)
        self.class_names = ["malignant", "benign"]

    def predict(self, context, model_input):
        """
        Returns a DataFrame with columns:
          - prediction: 0 or 1
          - label:      'malignant' or 'benign'
          - prob_malignant: float probability of class 0
          - prob_benign:    float probability of class 1
        """
        preds  = self.pipeline.predict(model_input)
        probas = self.pipeline.predict_proba(model_input)
        return pd.DataFrame({
            "prediction":     preds,
            "label":          [self.class_names[p] for p in preds],
            "prob_malignant": probas[:, 0],
            "prob_benign":    probas[:, 1],
        })


# Serialise the fitted pipeline to a temp file so we can register it as an artifact
import pickle
pipeline_artifact_dir = tempfile.mkdtemp()
pipeline_pkl_path = os.path.join(pipeline_artifact_dir, "pipeline.pkl")
with open(pipeline_pkl_path, "wb") as f:
    pickle.dump(full_pipeline, f)

print(f"\nPickled pipeline saved to: {pipeline_pkl_path}")
print(f"Pickle size: {os.path.getsize(pipeline_pkl_path) / 1024:.1f} KB")


In [ ]:
# Log the pyfunc model to MLflow
with mlflow.start_run(run_name="pyfunc-cancer-pipeline") as pyfunc_run:
    mlflow.log_metrics(pipeline_metrics)
    mlflow.set_tags(get_repro_tags())
    mlflow.set_tag("model.class",   "CancerPipelineModel")
    mlflow.set_tag("pipeline.steps", "StandardScaler|SelectKBest(k=15)|GradientBoosting")

    # Build pyfunc signature — input is the raw feature DataFrame; output is our rich dict
    sample_output = pd.DataFrame({
        "prediction":     [0, 1],
        "label":          ["malignant", "benign"],
        "prob_malignant": [0.9, 0.1],
        "prob_benign":    [0.1, 0.9],
    })
    pyfunc_sig = infer_signature(X_test.head(2), sample_output)

    mlflow.pyfunc.log_model(
        artifact_path="cancer-pipeline-pyfunc",
        python_model=CancerPipelineModel(),
        artifacts={"pipeline": pipeline_pkl_path},  # bundled artifact
        signature=pyfunc_sig,
        input_example=X_test.head(3),
        code_paths=None,  # no extra source files needed for this simple wrapper
    )
    pyfunc_run_id = pyfunc_run.info.run_id

print(f"pyfunc model logged to run: {pyfunc_run_id[:8]}…")

# Verify local loading works before registering
loaded_pyfunc = mlflow.pyfunc.load_model(f"runs:/{pyfunc_run_id}/cancer-pipeline-pyfunc")
sample_input  = X_test.head(5).reset_index(drop=True)
sample_output_actual = loaded_pyfunc.predict(sample_input)
print("\nSample pyfunc predictions:")
print(sample_output_actual.to_string(index=True))


**What just happened?**
- **`CancerPipelineModel`** subclasses `mlflow.pyfunc.PythonModel` — the two required methods are `load_context` (runs once at server startup) and `predict` (runs per request).
- **`artifacts={"pipeline": pipeline_pkl_path}`** bundles the pickled sklearn Pipeline alongside the Python code — when MLflow loads this model anywhere, it first runs `load_context` to deserialise the pickle.
- **Rich output DataFrame** — returning labels + probabilities alongside the binary prediction is the production pattern: downstream systems can apply their own business thresholds without re-serving.
- **Local load + verify** before registering is a best practice: catch any pickling/import issues before they reach the registry or a serving endpoint.


## Step 5 · Register the pyfunc Pipeline and Prepare Serving

We register the pyfunc model under a separate registry name so it's clear this is the *serving artifact* (full pipeline including preprocessing) versus the raw model variants from Step 2.

**Serving URI format for `mlflow models serve`:**
```bash
mlflow models serve \
  --model-uri "models:/breast-cancer-pipeline-pyfunc@champion" \
  --port 5001 \
  --no-conda
```

The endpoint accepts POST requests with JSON body:
```json
{"dataframe_split": {"columns": [...], "data": [[...], [...]]}}
```


In [ ]:
PYFUNC_REGISTRY_NAME = "breast-cancer-pipeline-pyfunc"

# Register the pyfunc model
pyfunc_uri = f"runs:/{pyfunc_run_id}/cancer-pipeline-pyfunc"
mv_pyfunc = mlflow.register_model(model_uri=pyfunc_uri, name=PYFUNC_REGISTRY_NAME)
print(f"Registered pyfunc pipeline → v{mv_pyfunc.version}")

# Add description and promote to Production
client.update_model_version(
    name=PYFUNC_REGISTRY_NAME,
    version=mv_pyfunc.version,
    description=(
        "Production pyfunc pipeline: StandardScaler + SelectKBest(k=15) + GradientBoosting. "
        f"roc_auc={pipeline_metrics['roc_auc']:.4f}. "
        f"Returns prediction, label, prob_malignant, prob_benign."
    ),
)
client.transition_model_version_stage(
    name=PYFUNC_REGISTRY_NAME,
    version=mv_pyfunc.version,
    stage="Production",
)
client.set_registered_model_alias(
    name=PYFUNC_REGISTRY_NAME, alias="champion", version=mv_pyfunc.version
)

serving_uri  = f"models:/{PYFUNC_REGISTRY_NAME}@champion"
print(f"Serving URI: {serving_uri}")

# Build the JSON payload that `mlflow models serve` expects
sample_rows  = X_test.head(3).reset_index(drop=True)
serving_payload = {
    "dataframe_split": {
        "columns": list(sample_rows.columns),
        "data": sample_rows.values.tolist(),
    }
}

print("\nSample REST payload (first 100 chars):")
payload_str = json.dumps(serving_payload)
print(payload_str[:120] + "…")

# Save the payload to a file so we can use it with curl / requests later
with open("/tmp/sample_payload.json", "w") as f:
    json.dump(serving_payload, f, indent=2)
print("\nFull payload saved to /tmp/sample_payload.json")


**What just happened?**
- We registered the pyfunc pipeline under a **separate registry name** (`breast-cancer-pipeline-pyfunc`) — this is the serving artifact, distinct from the raw model variants.
- **`dataframe_split` format** is what MLflow Serving expects for tabular data — it includes the column names alongside the data rows, so the server can reconstruct the DataFrame correctly.
- We serialised the payload to disk so you can verify it manually with `curl -X POST http://localhost:5001/invocations -H 'Content-Type: application/json' -d @/tmp/sample_payload.json`.


## Step 6 · Simulate the Serving Endpoint (Local Verification)

Running `mlflow models serve` requires a shell process and is not directly possible inside a Colab cell. Instead, we simulate the serving call by loading the pyfunc model and exercising the exact same code path that the REST endpoint uses.

**Production equivalent (run in a terminal):**
```bash
# Start the server
MLFLOW_TRACKING_URI=sqlite:///capstone_pipeline.db \
  mlflow models serve \
    --model-uri "models:/breast-cancer-pipeline-pyfunc@champion" \
    --port 5001 \
    --no-conda

# Call the endpoint
curl -X POST http://localhost:5001/invocations \
  -H 'Content-Type: application/json' \
  -d @/tmp/sample_payload.json
```

The simulation below mirrors what the MLflow server does internally:


In [ ]:
import requests as _requests_module  # imported to show the real REST pattern

def simulate_rest_invocation(model_uri: str, payload: dict) -> pd.DataFrame:
    """
    Simulate a POST /invocations call without starting a real server.
    Mirrors what mlflow models serve does internally:
      1. Load the pyfunc model from the registry
      2. Deserialise the dataframe_split payload into a DataFrame
      3. Call model.predict(df) and return the result
    """
    # 1. Load model (mlflow models serve does this once at startup)
    model = mlflow.pyfunc.load_model(model_uri)

    # 2. Deserialise payload (mlflow server parses the JSON body)
    ds = payload["dataframe_split"]
    input_df = pd.DataFrame(ds["data"], columns=ds["columns"])

    # 3. Predict (same method the server calls)
    return model.predict(input_df)


# Run the simulation
print("=== Simulated REST /invocations call ===")
print(f"Model URI: {serving_uri}")
print(f"Input rows: {len(serving_payload['dataframe_split']['data'])}")
print()

result_df = simulate_rest_invocation(serving_uri, serving_payload)
print("Response (as the REST endpoint would return):")
print(result_df.to_string(index=True))

# Cross-check: verify the predictions match raw sklearn output
raw_preds = full_pipeline.predict(sample_rows)
match = (result_df["prediction"].values == raw_preds).all()
print(f"\nPredictions match raw sklearn output: {match}")

# Show what the actual curl command would look like
print("\nEquivalent curl command:")
print("  mlflow models serve \\")
print(f"    --model-uri '{serving_uri}' \\")
print("    --port 5001 --no-conda")
print("  # then:")
print("  curl -X POST http://localhost:5001/invocations \\")
print("    -H 'Content-Type: application/json' \\")
print("    -d @/tmp/sample_payload.json")


**What just happened?**
- **`simulate_rest_invocation`** mirrors the exact three steps MLflow's serving engine performs: load model → parse payload → call `model.predict(df)`.
- The `dataframe_split` format is one of three MLflow input formats; the others are `dataframe_records` (list of row dicts) and `instances` (TensorFlow/numpy tensors).
- **Cross-checking against `full_pipeline.predict`** confirms the pyfunc wrapper produces identical binary predictions — the only difference is the richer output schema.
- In a real deployment, you'd run the `mlflow models serve` command in a Docker container or Kubernetes pod, not in a notebook.


## Step 7 · Log the Pipeline Report Run

The final step in a reproducible MLOps pipeline is a **summary run** — a special MLflow run that logs the entire pipeline's training history, champion selection, and serving configuration as a single auditable artifact.

This makes the end-to-end pipeline fully queryable: anyone can load this run to see what was trained, why the champion was chosen, and where it's served.


In [ ]:
import datetime

# Build the pipeline report as a JSON artifact
pipeline_report = {
    "report_generated_at": datetime.datetime.utcnow().isoformat() + "Z",
    "experiment_name":     EXPERIMENT_NAME,
    "training_history": [
        {
            "model": c["name"],
            "run_id": c["run_id"],
            "metrics": c["metrics"],
        }
        for c in candidates
    ],
    "champion": {
        "model":             champion_name,
        "run_id":            champion_run_id,
        "registry_name":     MODEL_REGISTRY_NAME,
        "version":           int(champion_mv.version),
        "alias":             "champion",
        "metrics":           champion_metrics,
        "promoted_at":       datetime.datetime.utcnow().isoformat() + "Z",
    },
    "pyfunc_pipeline": {
        "registry_name": PYFUNC_REGISTRY_NAME,
        "version":       int(mv_pyfunc.version),
        "serving_uri":   serving_uri,
        "steps":         ["StandardScaler", "SelectKBest(k=15)", "GradientBoostingClassifier"],
        "output_columns": ["prediction", "label", "prob_malignant", "prob_benign"],
        "endpoint": "http://localhost:5001/invocations",
        "payload_format": "dataframe_split",
    },
    "leaderboard": leaderboard[["model", "roc_auc", "accuracy", "f1"]].to_dict(orient="records"),
}

# Write report to a temp file
report_path = "/tmp/pipeline_report.json"
with open(report_path, "w") as f:
    json.dump(pipeline_report, f, indent=2)

print("Pipeline report written to:", report_path)
print(json.dumps({k: pipeline_report[k] for k in ["report_generated_at", "champion"]}, indent=2))


In [ ]:
# Log the summary run to MLflow
with mlflow.start_run(run_name="pipeline-report") as report_run:
    # Top-level metrics for easy querying
    mlflow.log_metrics({
        "champion_roc_auc":         champion_metrics["roc_auc"],
        "champion_accuracy":        champion_metrics["accuracy"],
        "champion_f1":              champion_metrics["f1"],
        "pyfunc_roc_auc":           pipeline_metrics["roc_auc"],
        "pyfunc_accuracy":          pipeline_metrics["accuracy"],
        "n_candidates_trained":     float(len(candidates)),
    })

    # Params capture the pipeline configuration
    mlflow.log_params({
        "champion_model":        champion_name,
        "champion_version":      str(champion_mv.version),
        "pyfunc_serving_uri":    serving_uri,
        "serving_endpoint":      "http://localhost:5001/invocations",
        "pipeline_steps":        "StandardScaler|SelectKBest(k=15)|GradientBoosting",
    })

    # Tags for discoverability
    mlflow.set_tags({
        **get_repro_tags(),
        "pipeline.stage":     "summary",
        "report.type":        "pipeline_report",
        "champion.run_id":    champion_run_id,
        "pyfunc.run_id":      pyfunc_run_id,
    })

    # Log the JSON report as an artifact
    mlflow.log_artifact(report_path, artifact_path="reports")

    # Log leaderboard as a CSV artifact for easy viewing
    leaderboard_path = "/tmp/leaderboard.csv"
    leaderboard.to_csv(leaderboard_path, index=False)
    mlflow.log_artifact(leaderboard_path, artifact_path="reports")

    report_run_id = report_run.info.run_id

print(f"Summary run logged: {report_run_id[:8]}…")
print("Artifacts:")
artifacts = client.list_artifacts(report_run_id, path="reports")
for a in artifacts:
    print(f"  {a.path}  ({a.file_size} bytes)")


**What just happened?**
- The **summary run** acts as an index for the entire pipeline execution — a single run_id that ties together all training runs, the registry promotion, and the serving configuration.
- **Logging top-level metrics** on the summary run means your CI dashboard can query `experiment_name + tags.report.type = 'pipeline_report'` to find the latest pipeline summary without knowing individual run IDs.
- **`log_artifact`** accepts any file — the JSON report and CSV leaderboard become versioned artifacts, accessible via the MLflow UI or `client.download_artifacts(run_id, ...)` in any downstream system.
- This pattern scales to automated pipelines: the summary run is created last by the orchestrator (Airflow, Prefect, GitHub Actions) and its run_id is the single handle for the entire pipeline execution.


## Step 8 · Final Validation — Load Champion and Run Inference

A pipeline is not complete until you verify the full serving path end-to-end:
1. Load the `@champion` pyfunc model from the registry (simulates what `mlflow models serve` does at startup)
2. Run inference on a held-out batch (simulates a production API call)
3. Confirm predictions are consistent with the training-time results


In [ ]:
print("=== End-to-End Validation ===")
print(f"Loading model from registry: {serving_uri}")

# Load the champion pyfunc model (same URI your serve command uses)
production_model = mlflow.pyfunc.load_model(serving_uri)

# Prepare a batch of 10 samples (unseen during training — from the test set)
validation_batch = X_test.tail(10).reset_index(drop=True)
validation_labels = y_test.tail(10).reset_index(drop=True)

# Predict
predictions = production_model.predict(validation_batch)

# Evaluate
val_acc = accuracy_score(validation_labels, predictions["prediction"])
val_f1  = f1_score(validation_labels, predictions["prediction"], zero_division=0)

print(f"\nValidation batch size : {len(validation_batch)}")
print(f"Batch accuracy        : {val_acc:.4f}")
print(f"Batch F1              : {val_f1:.4f}")
print("\nDetailed predictions:")
summary = pd.concat([
    predictions.reset_index(drop=True),
    pd.Series(validation_labels.values, name="true_label"),
], axis=1)
summary["correct"] = summary["prediction"] == summary["true_label"]
print(summary.to_string(index=True))

print("\nFull test-set classification report:")
all_preds = production_model.predict(X_test)["prediction"]
print(classification_report(y_test, all_preds, target_names=["malignant", "benign"]))


**What just happened?**
- We **loaded the model by alias** (`@champion`) from the registry — this is the production serving pattern. If you swap the champion to a new version, this code continues to work without any changes.
- **`tail(10)`** uses the last 10 test samples — ensuring we use data the model has never seen during training or the 5-fold CV.
- **`classification_report`** gives precision/recall/F1 per class — the most complete single-number summary for a binary classifier. For medical data, recall for the malignant class is the critical metric (you want to catch all malignant cases).
- This cell is your **acceptance test**: if the reported metrics match the training-time metrics within expected variance, the pipeline is deployment-ready.


In [ ]:
# Final pipeline summary
print("=" * 60)
print("  CAPSTONE PIPELINE SUMMARY")
print("=" * 60)
print(f"Experiment      : {EXPERIMENT_NAME}")
print(f"Models trained  : {len(candidates)} (LR, RF, GB)")
print()
print("Leaderboard (by roc_auc):")
print(leaderboard[["model", "roc_auc", "cv_roc_auc_mean", "accuracy", "f1"]].to_string(index=False))
print()
print(f"Champion        : {champion_name}")
print(f"  Registry      : {MODEL_REGISTRY_NAME} v{champion_mv.version} (@champion)")
print(f"  roc_auc       : {champion_metrics['roc_auc']:.4f}")
print(f"  accuracy      : {champion_metrics['accuracy']:.4f}")
print()
print(f"Pyfunc pipeline : {PYFUNC_REGISTRY_NAME} v{mv_pyfunc.version} (@champion)")
print(f"  Serving URI   : {serving_uri}")
print(f"  Pipeline      : StandardScaler → SelectKBest(k=15) → GradientBoosting")
print(f"  roc_auc       : {pipeline_metrics['roc_auc']:.4f}")
print()
print(f"Summary run     : {report_run_id[:8]}… (artifacts: pipeline_report.json, leaderboard.csv)")
print()
print("To serve in production:")
print(f"  MLFLOW_TRACKING_URI=sqlite:///capstone_pipeline.db \\")
print(f"  mlflow models serve \\")
print(f"    --model-uri '{serving_uri}' \\")
print(f"    --port 5001 --no-conda")
print("=" * 60)


In [ ]:
# Challenge: Extend the pipeline with automated rollback
#
# Part A — Rollback function:
#   Write `rollback_champion(registry_name, client)` that:
#     1. Finds the version currently aliased as @champion
#     2. Finds the version aliased as @previous (if it exists)
#     3. Swaps them: @previous becomes @champion, @champion becomes @previous
#     4. Returns the new champion version number, or raises ValueError if @previous doesn't exist
#
# Part B — Acceptance gate:
#   Write `acceptance_gate(pyfunc_uri, X_val, y_val, min_roc_auc=0.95)` that:
#     1. Loads the model from pyfunc_uri
#     2. Predicts on X_val and computes roc_auc using prob_benign column
#     3. Returns True if roc_auc >= min_roc_auc, False otherwise
#     4. Logs the gate result and threshold as a new MLflow run
#
# Hints for Part A:
#   - client.get_model_version_by_alias(registry_name, "champion") → ModelVersion
#   - client.get_model_version_by_alias(registry_name, "previous") → may raise MlflowException
#   - client.set_registered_model_alias(registry_name, alias, version)
#
# Hints for Part B:
#   - mlflow.pyfunc.load_model(pyfunc_uri).predict(X_val) returns a DataFrame
#   - Use the 'prob_benign' column for roc_auc_score
#   - from sklearn.metrics import roc_auc_score

def rollback_champion(registry_name, client):
    # Your solution here
    pass

def acceptance_gate(pyfunc_uri, X_val, y_val, min_roc_auc=0.95):
    # Your solution here
    pass

# Test rollback (uncomment after implementing):
# new_champ_ver = rollback_champion(PYFUNC_REGISTRY_NAME, client)
# print(f"Rolled back to v{new_champ_ver}")

# Test acceptance gate (uncomment after implementing):
# passed = acceptance_gate(serving_uri, X_test, y_test, min_roc_auc=0.95)
# print(f"Acceptance gate passed: {passed}")


---
## Day 14 key concepts recap

| Concept | What to remember |
|---|---|
| SQLite tracking URI | `sqlite:///file.db` — enables full Registry without a server; swap for PostgreSQL in production |
| Three-variant training | Each model in its own run: params + metrics + signature + repro tags |
| Champion selection | Rank by cv_roc_auc_mean (robust) not just test accuracy (optimistic) |
| `register_model` all variants | Register even non-champions — the registry is the audit trail |
| `@champion` alias | Serving code uses aliases, never version numbers — atomic swap during promotion |
| `mlflow.pyfunc.PythonModel` | Wrap any pipeline in `load_context` + `predict`; bundle artifacts dict |
| `dataframe_split` format | The JSON payload format for `POST /invocations` — includes column names |
| Summary report run | Final MLflow run logs training history + champion metrics + serving URI as artifacts |
| `classification_report` | Per-class precision/recall/F1 — for medical data, recall(malignant) is the key metric |
| Reproducibility tags | git.commit + data hash + library versions on every run — non-negotiable in production |

> **Tip:** The capstone is intentionally open-ended — a real MLOps pipeline should be reproducible from `git clone + mlflow run`. Use MLproject to make it one command.

---
## Capstone complete — what you built

1. Trained **3 model variants** (LogisticRegression, RandomForest, GradientBoosting) with full MLflow tracking
2. **Selected the champion** by ROC-AUC and CV score, registered all variants in the Model Registry
3. Promoted champion to **Production** with `@champion` alias and archived the rest
4. Built a **pyfunc Pipeline** (scaler + feature selector + model) with rich output schema
5. **Simulated REST serving** via `simulate_rest_invocation` — equivalent to `mlflow models serve`
6. Logged a **pipeline report run** with training history, champion metrics, and serving URI as artifacts
7. Ran **end-to-end validation** confirming predictions are consistent from training to serving

---
## You've completed the MLflow for ML Engineers course!

You now know how to:
- Track experiments, parameters, metrics, and artifacts
- Use the Model Registry for versioning and promotion
- Build reproducible pipelines with parent/child runs and repro tags
- Package models as pyfunc for flexible serving
- Apply champion/challenger patterns for safe production promotion

Mark Day 14 complete in your [tracker](../index.html).
